# 🚀 RLCS Automathon : votre premier agent

Ce notebook entraîne un agent par renforcement (PPO), le teste contre les bots fournis, puis prépare le zip
à envoyer dans **l'Arène**, l'application du tournoi. Comptez une dizaine de minutes une fois le cluster démarré.
Pour un premier essai, ne modifiez rien.

**Avant de commencer :**
- ce notebook est dans le **dossier de votre équipe** (pas dans votre dossier personnel) ;
- il est attaché au **cluster de votre équipe** (sélecteur en haut à droite) ;
- exécutez les cellules **dans l'ordre** avec `Maj + Entrée`.

## 1. Installer les bibliothèques

Les mêmes versions que dans l'Arène : un modèle entraîné ici s'y charge à l'identique. Environ 30 secondes,
à refaire après chaque redémarrage du cluster.

In [ ]:
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu torch==2.14.1 stable-baselines3==2.9.2 gymnasium==1.4.0 pydantic==2.14.1 pyzmq==27.2.0 numpy==2.2.6 numba==0.68.1 tqdm

In [ ]:
dbutils.library.restartPython()

## 2. Paramètres

Deux champs apparaissent en haut du notebook :
- **Équipe** : l'identifiant donné par les organisateurs (ex. `equipe_07`) ;
- **Membre** : votre numéro dans l'équipe, de 1 à 4. Le cluster est partagé : ce numéro vous réserve des ports.

In [ ]:
dbutils.widgets.text("equipe", "", "1. Équipe")
dbutils.widgets.dropdown("membre", "1", ["1", "2", "3", "4"], "2. Membre")

In [ ]:
import glob, importlib, os, platform, re, shutil, sys, time, zipfile
from datetime import datetime

# --- Fourni par les organisateurs ------------------------------------------------------------
BINAIRES = "/Volumes/automathon/partage/binaires"     # kits du jeu (lecture seule)
KITS = {"arm": "Automathon_1.0.0_Linux-ARM-databricks.zip",   # Headless + kit Python, processeurs ARM
        "x86": "Automathon_1.0.0_Linux-x86-64.zip"}           # le même pour les processeurs x86
VOLUMES_EQUIPES = "/Volumes/automathon/equipes"       # un Volume par équipe
URL_ARENE = "https://automathon-arena-7474646720384619.aws.databricksapps.com"
# ----------------------------------------------------------------------------------------------

EQUIPE = re.sub(r"[^a-z0-9_]", "_", dbutils.widgets.get("equipe").strip().lower())
assert EQUIPE, "Renseignez le champ « 1. Équipe » en haut du notebook, puis relancez la cellule."
MEMBRE = int(dbutils.widgets.get("membre"))
PORT_BASE = 5600 + 1000 * (MEMBRE - 1)              # membre 1 : 5600-5999, membre 2 : 6600-6999...
os.environ["AUTOMATHON_PORT_BASE"] = str(PORT_BASE)

VOLUME_EQUIPE = f"{VOLUMES_EQUIPES}/{EQUIPE}"         # vos modèles et vos zips : conservés
LOCAL = f"/tmp/automathon_{MEMBRE}"                   # disque local du cluster : effacé au redémarrage
assert os.path.isdir(VOLUME_EQUIPE), f"Volume introuvable : {VOLUME_EQUIPE}. Vérifiez l'identifiant d'équipe."
for dossier in ("modeles", "checkpoints", "soumissions"):
    os.makedirs(f"{VOLUME_EQUIPE}/{dossier}", exist_ok=True)
print(f"Équipe {EQUIPE}, membre {MEMBRE}. Volume : {VOLUME_EQUIPE}")

## 3. Le jeu sur le cluster

Décompresse le **Headless** (le jeu sans affichage, bien plus rapide que le temps réel) et le kit Python 1.0.0
sur le disque local, dans la version qui correspond au processeur du cluster. Puis une partie au hasard
contre `rush_bot`, pour vérifier que tout marche.

In [ ]:
ARCH = platform.machine()
KIT = KITS["arm"] if ARCH in ("aarch64", "arm64") else KITS["x86"]
JEU = f"{LOCAL}/jeu/{KIT[:-4]}"                                   # un dossier par version du kit
if not glob.glob(f"{JEU}/**/PythonAI/bridge", recursive=True):
    assert os.path.exists(f"{BINAIRES}/{KIT}"), f"{KIT} introuvable dans {BINAIRES} : prévenez un organisateur."
    zipfile.ZipFile(f"{BINAIRES}/{KIT}").extractall(JEU)
HEADLESS = [p for p in glob.glob(f"{JEU}/**/Headless/Headless", recursive=True) if os.path.isfile(p)][0]
SDK = os.path.dirname(glob.glob(f"{JEU}/**/PythonAI/bridge", recursive=True)[0])
for f in os.listdir(os.path.dirname(HEADLESS)):
    os.chmod(os.path.join(os.path.dirname(HEADLESS), f), 0o755)
os.environ["AUTOMATHON_HEADLESS_PATH"] = HEADLESS
if SDK not in sys.path:
    sys.path.insert(0, SDK)
try:
    VERSION = open(os.path.join(os.path.dirname(SDK), "VERSION")).read().strip()
except OSError:
    VERSION = "?"

from core.env import AutomathonEnv
from training.utils import launch_headless

port = PORT_BASE + 300
headless = launch_headless(port, timeout=60)
env = AutomathonEnv(tcp_port=str(port))
env.set_opponent_by_name("rush_bot")
try:
    obs, _ = env.reset()
    terminated = truncated = False
    while not (terminated or truncated):        # une décision = 4 frames ; une partie finit avant 4 200 frames
        obs, reward, terminated, truncated, _ = env.step(env.action_space.sample())
    etat = env._last_state
    pv = lambda vaisseau: float(vaisseau.Health) if vaisseau else 0
    print(f"OK : kit {VERSION}, processeur {ARCH}, {os.cpu_count()} cœurs. Partie au hasard finie après "
          f"{etat.Frame} frames, PV {pv(etat.SelfShip):.0f} contre {pv(etat.EnemyShip):.0f}.")
finally:
    env.game.end_training()
    headless.kill()

## 4. Votre agent

Un agent, c'est un dossier avec un fichier `agent.py` qui contient une classe `ParticipantAgent` et une méthode
`get_action(state) -> AIAction`. Le nôtre est celui du modèle d'agent `my_rl_agent` du kit : un réseau de neurones.
`feature_extractor.py` résume l'état du jeu en 112 nombres, le réseau en déduit 8, convertis en commandes
(déplacement, visée, mitrailleuse, missile, bouclier, dash). Le réseau entraîné est dans `model.zip`, à côté.

Parti à droite, l'agent voit le jeu tourné de 180°, comme à l'entraînement : il n'a qu'un côté à apprendre.

Pour modifier l'agent, modifiez le code de cette cellule puis relancez-la.

In [ ]:
AGENT_CODE = '''
from pathlib import Path

import numpy as np
from stable_baselines3 import PPO

from bridge.datatypes import AIAction, FixedVector2, GameState
from .feature_extractor import extract_features, flip_action, starts_on_right, warmup


def vers_action(a) -> AIAction:
    # 8 valeurs dans [-1, 1] -> commandes du vaisseau (même conversion qu'à l'entraînement)
    return AIAction(
        MovingDirection=FixedVector2(float(a[0]), float(a[1])),
        AimingDirection=FixedVector2(float(a[2]), float(a[3])),
        MachineGun=bool(a[4] > 0), Missile=bool(a[5] > 0), Shield=bool(a[6] > 0), Dash=bool(a[7] > 0),
    )


class ParticipantAgent:
    def __init__(self):
        self.model = PPO.load(Path(__file__).parent / "model.zip", device="cpu")
        warmup()                      # compile l'extracteur (Numba) maintenant, pas pendant le premier coup

    def get_action(self, state: GameState) -> AIAction:
        flip = starts_on_right(state)     # parti à droite : le jeu est vu tourné de 180°, comme à l'entraînement
        action, _ = self.model.predict(extract_features(state, flip=flip), deterministic=True)
        action = np.asarray(action, dtype=np.float32)
        return vers_action(flip_action(action) if flip else action)
'''

AGENT_DIR = f"{SDK}/agents/{EQUIPE}"
os.makedirs(AGENT_DIR, exist_ok=True)
open(f"{AGENT_DIR}/__init__.py", "w").close()
with open(f"{AGENT_DIR}/agent.py", "w") as f:
    f.write(AGENT_CODE.lstrip())
shutil.copy(f"{SDK}/agents/my_rl_agent/feature_extractor.py", f"{AGENT_DIR}/feature_extractor.py")  # 112 valeurs
importlib.invalidate_caches()
print("Agent :", AGENT_DIR)

## 5. Entraîner avec PPO

PPO fait jouer votre agent dans plusieurs parties en parallèle (un Headless par partie), puis ajuste le réseau
pour gagner plus de récompense. **Tous vos essais passent par la cellule de réglages ci-dessous.**
Pendant l'entraînement, surveillez `ep_rew_mean` (récompense moyenne par partie) : elle doit monter.

In [ ]:
# --- Réglages : c'est ici que vous expérimentez ! ------------------------------------------
DEPART = "zero"        # "zero" : réseau neuf · "dernier" : votre dernier modèle (dossier modeles/ du Volume)
ADVERSAIRE = "rush_bot"  # dummy_bot, rush_bot, ou l'un de vos bots (un dossier de agents/ dans le kit)
TIMESTEPS = 200_000    # décisions cumulées sur toutes les parties (1 décision = 4 frames de jeu)
NB_ENVS = 4            # parties en parallèle. Le cluster est partagé : à quatre, 4 chacun au plus
RECOMPENSE = dict(time_penalty=0.001, damage_dealt_coef=0.01, pain_factor=0.001,   # réglages du kit 1.0.0
                  win_bonus=10.0, loss_penalty=10.0, draw_penalty=10.0)
PPO_PARAMS = dict(learning_rate=3e-4, n_steps=2048, batch_size=1024, gamma=0.99, ent_coef=0.0)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor


def make_env(port):
    """Une partie Headless = un environnement, dans son propre processus."""
    def _init():
        from core.env import AutomathonEnv
        from training.utils import launch_headless
        extract_features = importlib.import_module(f"agents.{EQUIPE}.feature_extractor").extract_features
        headless = launch_headless(port, timeout=60)
        env = AutomathonEnv(tcp_port=str(port), extractor_fn=extract_features, reward_kwargs=RECOMPENSE,
                            random_side=True)                    # côté tiré au hasard à chaque partie
        env.set_opponent_by_name(ADVERSAIRE)

        def close():
            for arreter in (env.game.end_training, headless.kill):
                try:
                    arreter()
                except Exception:
                    pass
        env.close = close
        return env
    return _init


class SauvegardeVolume(BaseCallback):
    """Copie le modèle dans le Volume toutes les 50 000 décisions : il survit à un redémarrage du cluster."""
    def __init__(self, tous_les=50_000):
        super().__init__()
        self.tous_les, self.prochain = tous_les, tous_les

    def _on_step(self):
        if self.num_timesteps >= self.prochain:
            self.prochain += self.tous_les
            self.model.save(f"{LOCAL}/checkpoint.zip")              # un zip s'écrit en local...
            shutil.copy(f"{LOCAL}/checkpoint.zip", f"{VOLUME_EQUIPE}/checkpoints/{RUN}_{self.num_timesteps}.zip")
        return True                                                  # ...puis se copie dans le Volume


RUN = f"{datetime.now():%m%d_%H%M}_m{MEMBRE}"
precedents = sorted(glob.glob(f"{VOLUME_EQUIPE}/modeles/*.zip"), key=os.path.getmtime)
assert DEPART in ("zero", "dernier"), 'DEPART vaut "zero" ou "dernier".'
depart = precedents[-1] if DEPART == "dernier" and precedents else None
if DEPART == "dernier" and not precedents:
    print("Aucun modèle dans le Volume pour l'instant : départ d'un réseau neuf.")

vec_env = VecMonitor(SubprocVecEnv([make_env(PORT_BASE + i) for i in range(NB_ENVS)]))
try:
    if depart:
        print("Départ :", os.path.basename(depart))
        model = PPO.load(depart, env=vec_env, device="cpu", verbose=1, **PPO_PARAMS)
    else:
        print("Départ : réseau neuf")
        model = PPO("MlpPolicy", vec_env, device="cpu", verbose=1, **PPO_PARAMS)
    debut = time.time()
    model.learn(total_timesteps=TIMESTEPS, callback=SauvegardeVolume())
    print(f"Entraînement terminé en {(time.time() - debut) / 60:.1f} min")
finally:
    vec_env.close()

model.save(f"{LOCAL}/model.zip")
shutil.copy(f"{LOCAL}/model.zip", f"{AGENT_DIR}/model.zip")
shutil.copy(f"{LOCAL}/model.zip", f"{VOLUME_EQUIPE}/modeles/{EQUIPE}_{RUN}.zip")
print("Modèle enregistré :", f"{VOLUME_EQUIPE}/modeles/{EQUIPE}_{RUN}.zip")

## 6. Tester contre les bots

Votre agent joue 10 parties contre chacun des deux bots du kit, `dummy_bot` et `rush_bot`, en changeant de côté au
hasard (V = victoires, N = nuls, D = défaites). Avec si peu de parties, le hasard compte : comparez deux modèles sur
au moins 21 parties avant de conclure.

In [ ]:
from evaluation.run_matchup import play_matchup

port = PORT_BASE + 200
headless = launch_headless(port, timeout=60)
env = AutomathonEnv(tcp_port=str(port), random_side=True)       # côté tiré au hasard à chaque partie
try:
    for nom in ["dummy_bot", "rush_bot"]:
        adversaire = importlib.import_module(f"agents.{nom}.agent").ParticipantAgent()
        mon_agent = importlib.import_module(f"agents.{EQUIPE}.agent").ParticipantAgent()
        _, v, d, n = play_matchup(env, mon_agent, adversaire, 10, verbose=False)
        print(f"contre {nom:<10} {v} V / {n} N / {d} D")
finally:
    env.game.end_training()
    headless.kill()

## 7. Préparer le zip pour l'Arène

Le zip contient `agent.py`, `feature_extractor.py` et `model.zip`. Avant de l'enregistrer, la cellule le vérifie
**comme le fait l'Arène** : elle le décompresse à part, charge l'agent et lui fait jouer une partie contre
`dummy_bot`, en mesurant le temps de réponse par coup. Dans l'Arène, la limite est de 5 ms par coup en moyenne
sur une partie : au-delà, la partie est perdue.

In [ ]:
nom = f"{EQUIPE}_{datetime.now():%m%d_%H%M}_m{MEMBRE}"
archive = f"{LOCAL}/{nom}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:          # un zip s'écrit en local...
    for fichier in ("agent.py", "feature_extractor.py", "model.zip"):
        zf.write(f"{AGENT_DIR}/{fichier}", fichier)

# Vérification comme dans l'Arène : zip décompressé à part, une partie contre dummy_bot
verif = f"{EQUIPE}_verif"
shutil.rmtree(f"{SDK}/agents/{verif}", ignore_errors=True)
zipfile.ZipFile(archive).extractall(f"{SDK}/agents/{verif}")
open(f"{SDK}/agents/{verif}/__init__.py", "w").close()
importlib.invalidate_caches()
agent = importlib.import_module(f"agents.{verif}.agent").ParticipantAgent()
temps = []
get_action = agent.get_action
def chronometre(state):
    t = time.perf_counter()
    action = get_action(state)
    temps.append((time.perf_counter() - t) * 1000)
    return action
agent.get_action = chronometre

port = PORT_BASE + 250
headless = launch_headless(port)
env = AutomathonEnv(tcp_port=str(port))
try:
    _, v, d, n = play_matchup(env, agent, importlib.import_module("agents.dummy_bot.agent").ParticipantAgent(), 1, verbose=False)
finally:
    env.game.end_training()
    headless.kill()
moyenne = sum(temps) / len(temps)
print(f"Test OK : 1 partie contre dummy_bot ({'gagnée' if v else 'nulle' if n else 'perdue'}), "
      f"{moyenne:.1f} ms par coup en moyenne, {max(temps):.1f} ms au pire")
if moyenne > 5:
    print("ATTENTION : plus de 5 ms par coup en moyenne, la limite de l'Arène. Allégez votre agent avant de l'envoyer.")

shutil.copy(archive, f"{VOLUME_EQUIPE}/soumissions/{nom}.zip")          # ...puis se copie dans le Volume
catalogue, schema, volume = VOLUME_EQUIPE.split("/")[2:5]
print(f"Zip prêt : {VOLUME_EQUIPE}/soumissions/{nom}.zip")
print(f"Téléchargez-le : Catalog › {catalogue} › {schema} › {volume} › soumissions › {nom}.zip › ⋮ › Download")
print(f"Puis envoyez-le dans l'Arène : {URL_ARENE}")

## 8. Envoyer votre agent dans l'Arène

1. Téléchargez le zip (le chemin est affiché ci-dessus : **Catalog** › Volume de l'équipe › `soumissions/` › ⋮ › **Download**).
2. Ouvrez **l'Arène** (lien affiché ci-dessus) et connectez-vous avec votre compte Databricks.
3. Onglet **Soumettre** : glissez le zip dans la zone (ou cliquez pour le choisir), puis **Envoyer l'agent**.
4. Dans **Mes soumissions**, la colonne **Test** passe de « Test en cours » à **« Validé · x ms par coup »**
   (ou « Échec » : survolez-le pour lire l'erreur).
5. Onglet **Replays** : regardez la partie de test, puis les matchs de votre équipe au ladder.
6. Onglet **Classement** : le ladder rejoue toutes les équipes toutes les heures.

Chaque envoi remplace le précédent : soumettez tôt, puis souvent. Après la fermeture des soumissions,
c'est votre dernière soumission validée qui joue le tournoi.

## Pour aller plus loin

- **Récompense** : changez les poids de `RECOMPENSE` (ex. `pain_factor` plus haut pour un agent plus prudent).
- **Adversaires** : commencez contre `dummy_bot`, puis `rush_bot`, puis vos propres bots (copiez-les dans `agents/` du kit) ; `DEPART = "dernier"` reprend votre dernier modèle.
- **Plus longtemps** : avec les réglages du kit, 3 millions de `TIMESTEPS` donnent environ 43 % de victoires contre `rush_bot`. Le cluster tourne pendant que vous préparez la suite.
- **Observations** : enrichissez l'extracteur (`feature_extractor.py` du dossier de votre agent, copié depuis `agents/my_rl_agent/`) et repartez de `"zero"` si la taille change.
- **Le jeu** : la zone sûre se referme de 10 s à 40 s (`state.ZoneHalfSize`), et un missile n'est armé qu'après 1 unité de vol (`Armed`) : voir `PythonAI/README.md` du kit.

## 🧹 Dépannage : « Le port … est déjà utilisé »

Un Headless d'une cellule précédente tourne encore. Cette cellule arrête **vos** Headless (ceux de votre plage
de ports), pas ceux de vos coéquipiers.

In [ ]:
import subprocess
subprocess.run(["pkill", "-f", f"tcp://127.0.0.1:{4 + MEMBRE}"])
print("Headless du membre", MEMBRE, "arrêtés.")